# Phase-Plane Widget for the Mean-Field (FS + RS) Model

This notebook demonstrates the interactive phase-plane widget for the 2-population mean-field model.

The widget renders:
- **Phase plane** with nullclines, vector field, fixed points, and trajectories
- **Time series** for each population's firing rate

All computations (nullclines, vector field, fixed points, trajectories) are performed in Python using the exact `MFModel.rhs()` pipeline, while the widget UI handles Canvas rendering and interactivity.

Use the **slider controls** to vary external input rates, the population time constant `τ_f`, or the transfer-function scaling factors `α_FS` and `α_RS`.

The built-in **Export SVG** button generates a vector figure of the current phase-plane view.

In [7]:
import sys
sys.version

'3.12.10 (main, Apr  9 2025, 04:03:51) [Clang 20.1.0 ]'

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import json
from ntmf.phase_plane_widget import PhasePlaneWidget
from ntmf.phase_plane import NTMFMeanField
from ntmf.config import get_network_config, get_params_model_SI, adding_K_params
from ntmf.validation import build_mf_network_config

In [4]:
# ── Build network config with external-input quantal conductances ──
network_config = build_mf_network_config(
    '../../config/network_config_file_v0.json',
    Q_e_nS=1.5, Q_i_nS=5.0,
)

# ── Neuron model parameters (SI units) ──
params_SI = {
    'FS': adding_K_params(
        get_params_model_SI('FS', '../../neuron_models/AdEx/FS.json').copy(),
        network_config,
    ),
    'RS': adding_K_params(
        get_params_model_SI('RS', '../../neuron_models/AdEx/RS.json').copy(),
        network_config,
    ),
}

# ── Load transfer-function fit parameters ──
with open('../../transfer_function/validation/FS_MF_params.json') as f:
    fs_fits = json.load(f)
with open('../../transfer_function/validation/RS_MF_params.json') as f:
    rs_fits = json.load(f)

fs_fit = fs_fits[0]   # first FS fit
rs_fit = rs_fits[0]   # first RS fit

poly_params = {
    'FS': fs_fit['polynomial_params'],
    'RS': rs_fit['polynomial_params'],
}
alphas = {
    'FS': fs_fit['alpha'],
    'RS': rs_fit['alpha'],
}

print(f"FS fit:  alpha = {alphas['FS']:.3f},  mean_error = {fs_fit.get('mean_error', 'n/a')}")
print(f"RS fit:  alpha = {alphas['RS']:.3f},  mean_error = {rs_fit.get('mean_error', 'n/a')}")

FS fit:  alpha = 1.165,  mean_error = 6.692889852581648
RS fit:  alpha = 1.170,  mean_error = 0.24510519542313824


In [5]:
# ── Build the MF → BaseModel adapter ──
mf_model = NTMFMeanField(
    params=params_SI,
    poly_params=poly_params,
    alphas=alphas,
    network_config=network_config,
    tau_f=0.01,
)

# ── Create the widget in Python-compute mode ──
widget = PhasePlaneWidget(
    model=mf_model,
    python_compute=True,
    xlim=[0, 80],
    ylim=[0, 80],
    t_max=100.0,
)

# The widget's initial computation is triggered automatically on display.

*small bug in the widget, it shows Models like Wilson Cowan but the model is actually the NTMFMeanField*

In [6]:
widget

## How it works

This widget is running in **Python-compute mode** (`python_compute=True`). Here's what happens on every slider change:

1. The JavaScript front-end detects the parameter change and syncs it to the Python kernel via `anywidget`.
2. Python's `PhasePlaneWidget._run_python_compute()` calls `BaseModel.compute_nullclines()`, `find_fixed_points()`, `compute_vector_field()`, and `compute_trajectory()` on the wrapped `NTMFMeanField` model.
3. These results are written to synced traitlets (e.g. `nullcline_x`, `fixed_points`, `trajectory`).
4. The traits sync back to the browser, and the JavaScript renderer repaints the Canvas with the fresh data.

### Key advantage
The `NTMFMeanField.f()` method calls the **exact same** `MFModel.rhs()` and `TF_template_sim()` functions used in production simulations. There is no approximation or model duplication.

### Exporting figures
Use the **Export SVG** button embedded in the widget to download a publication-quality vector rendering of the current phase plane (nullclines + vector field + fixed points + trajectory).